1 **load data**

In [31]:
import pandas as pd

df = pd.read_csv('source.csv')
df.head()

,Name,Datetime,Amount,Price,Purity
0,ProductA,2022-01-01T01:00:00.000,10,22.09,Impure
1,ProductA,2022-01-01T02:00:00.000,15,24.22,Pure
2,ProductA,2022-01-01T03:00:00.000,10,25.96,Impure
3,ProductA,2022-01-01T04:00:00.000,20,21.16,Impure
4,ProductA,2022-01-01T05:00:00.000,10,20.05,Pure


In [32]:
df.columns

Index(['Name', 'Datetime', 'Amount', 'Price', 'Purity'], dtype='object')

In [33]:
df['Datetime'].dtypes

dtype('O')

In [34]:
df.shape

(24, 5)

2 **Convert values in datetime column from timezone UTC to UTC+6**

In [35]:
df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True)
df['Datetime'] = df['Datetime'].dt.tz_convert('Asia/Almaty')

In [36]:
df.head()

,Name,Datetime,Amount,Price,Purity
0,ProductA,2022-01-01 07:00:00+06:00,10,22.09,Impure
1,ProductA,2022-01-01 08:00:00+06:00,15,24.22,Pure
2,ProductA,2022-01-01 09:00:00+06:00,10,25.96,Impure
3,ProductA,2022-01-01 10:00:00+06:00,20,21.16,Impure
4,ProductA,2022-01-01 11:00:00+06:00,10,20.05,Pure


In [37]:
df.tail(3)

,Name,Datetime,Amount,Price,Purity
21,ProductB,2022-01-01 16:00:00+06:00,20,22.67,Pure
22,ProductB,2022-01-01 17:00:00+06:00,10,21.99,Impure
23,ProductB,2022-01-01 18:00:00+06:00,20,21.72,Impure


In [38]:
df['Purity'].value_counts()

Purity
Impure    14
Pure      10
Name: count, dtype: int64

3. **Adding new colums**

 Prepare Product A and Product B

In [39]:
print(df.columns)

Index(['Name', 'Datetime', 'Amount', 'Price', 'Purity'], dtype='object')


In [40]:
product_a = df[df['Name'] == 'ProductA'][['Datetime', 'Price']].copy()
product_b = df[df['Name'] == 'ProductB'].copy()

In [41]:
product_a

,Datetime,Price
0,2022-01-01 07:00:00+06:00,22.09
1,2022-01-01 08:00:00+06:00,24.22
2,2022-01-01 09:00:00+06:00,25.96
3,2022-01-01 10:00:00+06:00,21.16
4,2022-01-01 11:00:00+06:00,20.05
5,2022-01-01 12:00:00+06:00,21.27
6,2022-01-01 13:00:00+06:00,20.08
7,2022-01-01 14:00:00+06:00,22.19
8,2022-01-01 15:00:00+06:00,21.23
9,2022-01-01 16:00:00+06:00,22.34


In [42]:
product_b

,Name,Datetime,Amount,Price,Purity
12,ProductB,2022-01-01 07:00:00+06:00,20,22.37,Impure
13,ProductB,2022-01-01 08:00:00+06:00,20,24.54,Pure
14,ProductB,2022-01-01 09:00:00+06:00,10,26.03,Impure
15,ProductB,2022-01-01 10:00:00+06:00,20,21.19,Impure
16,ProductB,2022-01-01 11:00:00+06:00,10,20.23,Pure
17,ProductB,2022-01-01 12:00:00+06:00,15,21.34,Impure
18,ProductB,2022-01-01 13:00:00+06:00,20,20.79,Pure
19,ProductB,2022-01-01 14:00:00+06:00,10,23.32,Pure
20,ProductB,2022-01-01 15:00:00+06:00,20,21.43,Impure
21,ProductB,2022-01-01 16:00:00+06:00,20,22.67,Pure


Merge Product B with Product A price based on datetime

In [43]:
merged_b = product_b.merge(
    product_a,
    on='Datetime',
    how='left',
    suffixes=('', '_ProductA')  # So 'Price_ProductA' is created
)

In [45]:
merged_b 

,Name,Datetime,Amount,Price,Purity,Price_ProductA
0,ProductB,2022-01-01 07:00:00+06:00,20,22.37,Impure,22.09
1,ProductB,2022-01-01 08:00:00+06:00,20,24.54,Pure,24.22
2,ProductB,2022-01-01 09:00:00+06:00,10,26.03,Impure,25.96
3,ProductB,2022-01-01 10:00:00+06:00,20,21.19,Impure,21.16
4,ProductB,2022-01-01 11:00:00+06:00,10,20.23,Pure,20.05
5,ProductB,2022-01-01 12:00:00+06:00,15,21.34,Impure,21.27
6,ProductB,2022-01-01 13:00:00+06:00,20,20.79,Pure,20.08
7,ProductB,2022-01-01 14:00:00+06:00,10,23.32,Pure,22.19
8,ProductB,2022-01-01 15:00:00+06:00,20,21.43,Impure,21.23
9,ProductB,2022-01-01 16:00:00+06:00,20,22.67,Pure,22.34


Combine A and merged B

In [46]:
combined_df = pd.concat([
    df[df['Name'] == 'ProductA'],
    merged_b
], ignore_index=True)

In [47]:
combined_df

,Name,Datetime,Amount,Price,Purity,Price_ProductA
0,ProductA,2022-01-01 07:00:00+06:00,10,22.09,Impure,NaN
1,ProductA,2022-01-01 08:00:00+06:00,15,24.22,Pure,NaN
2,ProductA,2022-01-01 09:00:00+06:00,10,25.96,Impure,NaN
3,ProductA,2022-01-01 10:00:00+06:00,20,21.16,Impure,NaN
4,ProductA,2022-01-01 11:00:00+06:00,10,20.05,Pure,NaN
5,ProductA,2022-01-01 12:00:00+06:00,10,21.27,Impure,NaN
6,ProductA,2022-01-01 13:00:00+06:00,20,20.08,Pure,NaN
7,ProductA,2022-01-01 14:00:00+06:00,10,22.19,Pure,NaN
8,ProductA,2022-01-01 15:00:00+06:00,15,21.23,Impure,NaN
9,ProductA,2022-01-01 16:00:00+06:00,20,22.34,Pure,NaN


 Total computation function

In [48]:
def compute_total(row):
    price = row['Price']
    amount = row['Amount']
    purity = row['Purity']
    product = row['Name']

    if product == 'ProductA':
        if purity == 'Impure':
            price *= 0.75
        return price * amount

    elif product == 'ProductB':
        base_price = row.get('Price_ProductA', None)
        if pd.isna(base_price):
            return 0
        if purity == 'Impure':
            price *= 0.75
            base_price *= 0.75
        return (price - base_price) * amount

    return 0

In [49]:
combined_df['total'] = combined_df.apply(compute_total, axis=1)

In [50]:
combined_df.head(3)

,Name,Datetime,Amount,Price,Purity,Price_ProductA,total
0,ProductA,2022-01-01 07:00:00+06:00,10,22.09,Impure,NaN,165.675
1,ProductA,2022-01-01 08:00:00+06:00,15,24.22,Pure,NaN,363.300
2,ProductA,2022-01-01 09:00:00+06:00,10,25.96,Impure,NaN,194.700


In [51]:
combined_df.tail(3)

,Name,Datetime,Amount,Price,Purity,Price_ProductA,total
21,ProductB,2022-01-01 16:00:00+06:00,20,22.67,Pure,22.34,6.60
22,ProductB,2022-01-01 17:00:00+06:00,10,21.99,Impure,21.87,0.90
23,ProductB,2022-01-01 18:00:00+06:00,20,21.72,Impure,21.71,0.15


 Drop helper column

In [52]:
if 'Price_ProductA' in combined_df.columns:
    combined_df.drop(columns=['Price_ProductA'], inplace=True)

Save to result.csv

In [53]:
combined_df.to_csv('result.csv', index=False)